# 13 — Local transformer entity extraction

Explore **GLiNER small v2.1 zero-shot extraction** on five entity types: KNOWLEDGE, SKILL, TITLE, DEGREE and **EXPERIENCE_YEARS**. The rules comparator in notebook 11 supports only the original four types. This is a separate local model experiment, not a validated improvement over rules. The approved [transformer plan](../plan/2026-10-06-transformer-entity-extraction/README.md) authorizes a five-CV/five-JD smoke test followed by a bounded 100-CV/100-JD CPU run.

The default run samples only the frozen A2 training region at seed 0, uses a declared threshold of 0.5 and caps exploratory inference at 30 minutes. Compatible local results reload without repeating inference. A partial run must be read against its reported completion and coverage, not as a completed 200-document evaluation.

All inference is local. Real neural spans stay in ignored parquet because a predicted type cannot guarantee that a span contains no personal name. Default saved entity previews use authored synthetic text only.

## Years of experience

The approved [experience-years plan](../plan/2026-10-09-experience-years/README.md) adds explicit professional duration mentions, such as **5 years of experience**, **3–5 years**, or **at least 2 years**. Original wording and offsets are retained; ranges and minimums are not converted into exact totals. CV claims and JD requirements remain distinguished by `doc_type` and source context.

This does not calculate tenure from employment dates or add overlapping jobs. Age, calendar years and education duration are outside the intended label. Predictions can still be wrong and require independent evaluation. Years of experience can support future fit classification, but mentions alone do not establish fit; a small minimum-years gap can still be Potential Fit under the annotation guide.

The model prompt is `years of experience`. Synthetic probes showed the longer description returned no duration spans; the shorter prompt detected the CV example at the unchanged 0.5 threshold. Some JD range/minimum examples were still missed. This is exploratory extraction with incomplete recall, not a validated duration parser.


In [1]:
import json
from pathlib import Path
import pandas as pd
from candidate_screener.config import PROCESSED

# Keep every saved output below the notebook's 2,000-character cap.
def show_summary(value):
    rendered = json.dumps(value, indent=2, default=str)
    print(rendered if len(rendered) <= 1900 else rendered[:1850] + "\n[output truncated]")

MODEL_DIR = PROCESSED / "extraction" / "models" / "gliner-small-v2.1"
SAMPLE_SIZE = 100  # per type; configurable after measuring this exploratory run
SEED = 0
THRESHOLD = 0.5
MAX_SECONDS = 1800


## Run or reload the real local experiment

The package verifies checkpoint artifacts and input provenance, preserves field-relative offsets through overlapping tokenizer windows, merges duplicate window predictions and writes model results separately from rules. Missing dependencies or model artifacts fail explicitly; they never silently substitute rule outputs.

The run manifest records checkpoint revision, hashes, labels, threshold, chunk policy, heuristic context version, resolved dependencies and device. Scores are finite **raw, uncalibrated model scores**; neither a score nor an extracted mention proves proficiency or job fit.

In [2]:
from candidate_screener.extraction.transformer_run import run_transformer, validate_transformer_run

result = run_transformer(sample_size=SAMPLE_SIZE, seed=SEED, threshold=THRESHOLD,
                         max_seconds=MAX_SECONDS, model_dir=MODEL_DIR)
show_summary({"run_id": result["run_id"], "local_evidence_directory": result["run_dir"]})


{
  "run_id": "4fde336ec50802f910d2-20261009T161236977778Z",
  "local_evidence_directory": "/Users/user/Downloads/MTECH2526/PLP_project/intelligent-candidate-screener/data/processed/extraction/transformer-runs/4fde336ec50802f910d2-20261009T161236977778Z"
}


In [3]:
show_summary({key: result["summary"][key] for key in ("status", "runtime_seconds", "requested_documents", "completed_documents", "gold_accuracy")})

{
  "status": "complete",
  "runtime_seconds": 174.45759062599973,
  "requested_documents": 200,
  "completed_documents": 200,
  "gold_accuracy": {
    "status": "blocked",
    "reason": "No validated local NER gold adapters"
  }
}


In [4]:
show_summary(result["summary"]["coverage"]["cv"] )

{
  "requested_documents": 100,
  "completed_documents": 100,
  "documents_with_entities": 100,
  "entities": 1093,
  "by_label": {
    "KNOWLEDGE": 828,
    "SKILL": 30,
    "TITLE": 204,
    "DEGREE": 20,
    "EXPERIENCE_YEARS": 11
  },
  "documents_with_label": {
    "KNOWLEDGE": 80,
    "SKILL": 16,
    "TITLE": 92,
    "DEGREE": 12,
    "EXPERIENCE_YEARS": 9
  },
  "fields_requested": 400,
  "fields_attempted": 400,
  "fields_complete": 400,
  "fields_incomplete": 0,
  "quarantined_predictions": 0,
  "raw_uncalibrated_scores": {
    "min": 0.5000355839729309,
    "max": 0.98607337474823,
    "mean": 0.7253614810771716
  }
}


In [5]:
show_summary(result["summary"]["coverage"]["jd"] )

{
  "requested_documents": 100,
  "completed_documents": 100,
  "documents_with_entities": 99,
  "entities": 754,
  "by_label": {
    "KNOWLEDGE": 510,
    "SKILL": 71,
    "TITLE": 145,
    "DEGREE": 23,
    "EXPERIENCE_YEARS": 5
  },
  "documents_with_label": {
    "KNOWLEDGE": 82,
    "SKILL": 43,
    "TITLE": 88,
    "DEGREE": 17,
    "EXPERIENCE_YEARS": 5
  },
  "fields_requested": 200,
  "fields_attempted": 200,
  "fields_complete": 200,
  "fields_incomplete": 0,
  "quarantined_predictions": 0,
  "raw_uncalibrated_scores": {
    "min": 0.5002889633178711,
    "max": 0.9670681357383728,
    "mean": 0.6820634856464376
  }
}


In [6]:
show_summary(result["summary"]["rule_comparison"])

{
  "interpretation": "Exact span-label disagreement, not accuracy",
  "completed_documents_compared": 200,
  "cv": {
    "rule_entities": 404,
    "transformer_entities": 1093,
    "exact_shared": 244,
    "rule_only": 160,
    "transformer_only": 849
  },
  "jd": {
    "rule_entities": 327,
    "transformer_entities": 754,
    "exact_shared": 194,
    "rule_only": 133,
    "transformer_only": 560
  }
}


## Rule comparison and operational validation

The rule baseline is computed on the **identical processed document subset**. Counts and disagreement show differences in extraction behaviour; agreement does not create gold labels or establish superiority. Field coverage and quarantine must be considered alongside mention coverage, especially for incomplete runs.

In [7]:
integrity = validate_transformer_run(result)
show_summary(integrity)


{
  "entities_checked": 1847,
  "unique_entity_ids": true,
  "exact_field_offsets": true,
  "training_only": true,
  "artifact_hashes": true,
  "checkpoint_hashes": true,
  "field_coverage_valid": true,
  "status": "complete",
  "score_replay": "Not required; raw scores can vary across devices"
}


## Synthetic model evidence

This example is authored text. The package returns exact spans into its immutable original field. Display redaction is separate from evidence validation and `assert_clean` runs inside the notebook. This illustrates model/context behaviour; it is not an accuracy test. The context layer remains a versioned rule heuristic, so this experiment is a **hybrid neural mention extractor with heuristic context**.

In [8]:
from candidate_screener.extraction.transformer import load_model, extract_transformer_document
from candidate_screener.extraction.display import preview_entities
from candidate_screener.annotation.redact import assert_clean, redact_text

synthetic_text = "Software engineer with 5 years of professional experience and a Master of Science. Python."
synthetic_document = {"doc_id": "synthetic-transformer-demo", "source": "synthetic", "doc_type": "cv", "fields": {"Moreinfo": synthetic_text}}
model = load_model(MODEL_DIR, device="cpu")
synthetic_entities, synthetic_coverage = extract_transformer_document(
    synthetic_document, model, threshold=THRESHOLD, overlap_tokens=32)
assert all(synthetic_text[e["start"]:e["end"]] == e["surface"] for e in synthetic_entities)
assert any(e["label"] == "EXPERIENCE_YEARS" for e in synthetic_entities), "Synthetic experience duration was not extracted"
synthetic_preview = preview_entities(synthetic_entities, limit=5)
assert_clean(pd.Series([redact_text(row["surface"]) for row in synthetic_preview], dtype="string"), "synthetic transformer preview")
show_summary(synthetic_preview)

if callable(getattr(model, "close", None)):
    model.close()


[
  {
    "surface": "Software engineer",
    "label": "TITLE",
    "field": "Moreinfo",
    "context_status": "unqualified",
    "doc_type": "cv"
  },
  {
    "surface": "5 years of professional experience",
    "label": "EXPERIENCE_YEARS",
    "field": "Moreinfo",
    "context_status": "unqualified",
    "doc_type": "cv"
  },
  {
    "surface": "Master of Science",
    "label": "DEGREE",
    "field": "Moreinfo",
    "context_status": "unqualified",
    "doc_type": "cv"
  },
  {
    "surface": "Python",
    "label": "KNOWLEDGE",
    "field": "Moreinfo",
    "context_status": "unqualified",
    "doc_type": "cv"
  }
]


In [9]:
show_summary(synthetic_coverage)

{
  "Moreinfo": {
    "intervals": [
      [
        0,
        90
      ]
    ],
    "characters": 90,
    "chunks": 1,
    "planned_chunks": 1,
    "complete": true,
    "quarantined_intervals": [],
    "quarantined_predictions": 0,
    "interrupted": false
  }
}


## What this experiment can establish

This run can establish that a local transformer produces traceable predictions, its observed runtime and descriptive document/label coverage, and how its predictions differ from the limited seed rules. It does not establish precision, recall, F1, canonical mapping accuracy or a retrieval improvement.

DataTurks/SkillSpan restoration and validated adapters remain prerequisites for held-out extraction scoring. A later BERT/DistilBERT/DeBERTa token-classification experiment needs independently labelled spans, subword alignment, preserved splits/grouped folds and a training budget; rule predictions are not human gold. B1 automatically segmented skill blobs remain silver labels. No supervised training or new annotation runs here.

Inspect real records deliberately in the local `entities.parquet` at the evidence directory printed above. The saved-output helper suppresses unreviewed real neural spans rather than claiming regex redaction removes names. Any downstream retrieval comparison must report Recall@10 separately for `generic` and `targeted`; Precision@k must carry its ceiling and A13 uncertainty.